## ------------------------------------------------------------ ###
## Set up environment
## ------------------------------------------------------------ ###

In [2]:
import os
import sys
import glob
import rasterio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('../../ProcessEvents/')
from config import CATCHMENT_LOOKUP_DICT, OUT_DIR # MOLLY_DIR_FF, RAINFALL_CSV_DIR, ENSEMBLE_MEMBERS, , CATCHMENTS
event_details_fp = 'EventDetails_v4' # 'EventDetails_v5'
flood_5km_results_fp = "5km_total" # "5km_total_v5"

## ------------------------------------------------------------ ###
## Get list of catchments with all outputs
## ------------------------------------------------------------ ###

In [3]:
all_catchments = set(CATCHMENT_LOOKUP_DICT.keys())

catchments_with_flood_output = []
for catchment_num in all_catchments:
    catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
    fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/Catchment_{catchment_num}/{catchment_name}_new_Oct.pkl"
    flood_fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/PluvialResults/{flood_5km_results_fp}/Catchment_{catchment_num}/Ens01_{catchment_num}/10cm/flooded_area_5km_total_Ens01_{catchment_num}_10cm.nc"
        
    if os.path.isfile(flood_fp) and os.path.isfile(fp):
        catchments_with_flood_output.append(catchment_num)
    else:
        pass
        #print(catchment_num)
        #print(os.path.isfile(fp))
        #print(os.path.isfile(flood_fp))
        
# catchments_with_flood_output  = ['40', '23', '105']     
len(catchments_with_flood_output)

91

## ------------------------------------------------------------ ###
## Get list of catchments with all outputs
## ------------------------------------------------------------ ###

In [19]:
rainfall_events_all = []
for catchment_num in catchments_with_flood_output:
    catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
    combined_fp_new = os.path.join(OUT_DIR, f"Catchment_{catchment_num}", f"{catchment_name}_new_Oct.pkl")
    events_this_catchment = pd.read_pickle(combined_fp_new)

    events_this_catchment['catchment_num'] = catchment_num

    rainfall_events_all.append(events_this_catchment) 
rainfall_events_all_df = pd.concat(rainfall_events_all, ignore_index=True)   

len(rainfall_events_all_df.loc[~rainfall_events_all_df["catchment_num"].map(lambda x: isinstance(x, (int, float))), "catchment_num"].unique())

91

In [28]:
rainfall_events_all_df['rain_diff'] = rainfall_events_all_df['max_precip'] -  rainfall_events_all_df['max_precip_from_csv']
np.nanmax(rainfall_events_all_df['rain_diff'] )

np.float64(0.0004863281251346052)

In [26]:
rainfall_events_all_df[:50][['max_precip', 'max_precip_from_csv']]

,max_precip,max_precip_from_csv
0,33.561630,33.56163
1,36.918743,36.91874
2,32.719406,32.71941
3,41.163425,41.16343
4,30.752762,30.75276
5,38.064514,38.06451
6,33.925816,33.92582
7,37.368622,37.36862
8,33.426731,33.42673
9,33.182236,33.18224


In [23]:
list(rainfall_events_all_df.columns)

['max_precip',
 't_global',
 't_local',
 'x_idx',
 'y_idx',
 'x_idx_global',
 'y_idx_global',
 'x_coord',
 'y_coord',
 'time_mismatch',
 'ens',
 'start_idx',
 'stop_idx',
 'max_precip_from_csv',
 'start_year',
 'start_month',
 'start_day',
 'start_hour',
 'event_num',
 'event_durations',
 'rainfall_peak_day',
 'hydro_day_360',
 'day_360',
 't_local_rebased',
 'mismatch',
 'first_wet_idx',
 'trimmed_duration_hrs',
 '10cm_area',
 '10cm_volume',
 '30cm_area',
 '30cm_volume',
 'sm_day_before_onset',
 'sm_anchor_date',
 'neighbourhood_rain_total',
 'neighbourhood_rain_sum_excl_peak',
 'neighbourhood_n_cells',
 'neighbourhood_flood_10_area',
 'neighbourhood_flood_10_vol',
 'neighbourhood_flood_30_area',
 'neighbourhood_flood_30_vol',
 'threshold_rain_total_t50',
 'threshold_n_cells_t50',
 'cluster_rain_total_t50',
 'cluster_n_cells_t50',
 'threshold_flood_10_area_t50',
 'cluster_flood_10_area_t50',
 'threshold_flood_10_vol_t50',
 'cluster_flood_10_vol_t50',
 'threshold_flood_30_area_t50',
 '

## ------------------------------------------------------------ ###
## Check if any catchments have NANs for any variables
## ------------------------------------------------------------ ###

In [13]:
nulls = rainfall_events_all_df[rainfall_events_all_df['hc_at_peak'].isnull()]
np.unique(nulls)

array([], dtype=object)

In [14]:
nulls = rainfall_events_all_df[rainfall_events_all_df['fu_at_peak_new'].isnull()][['catchment_num', 'ens', 'fu_at_peak_new', 'lu_at_peak', 'event_num']]
nulls

,catchment_num,ens,fu_at_peak_new,lu_at_peak,event_num


## ------------------------------------------------------------ ###
## Check whether any grid cells have multiple events in them
## ------------------------------------------------------------ ###

In [15]:
dupes = rainfall_events_all_df[rainfall_events_all_df.duplicated(subset=["x_coord", "y_coord"],
        keep=False)].sort_values(["x_coord", "y_coord"])

len(dupes[['month', 'lu_at_peak', 'max_precip', 'ens', "x_coord", "y_coord"]])

106430

## ------------------------------------------------------------ ###
## Save to file
## ------------------------------------------------------------ ###

In [16]:
rainfall_events_all_df.to_pickle(f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/all_catchments.pkl")